# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.71 GB
MemFree: 550.00 GB
MemAvailable: 969.81 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successfu

# 2. Load CoQA Dataset

## 2.1 Download CoQA

In [16]:
import os
from pathlib import Path
import requests
import json
from tqdm import tqdm

def download_coqa_dataset(
    save_dir: str = "/nfs/students/daro/data/CoQA",
    version: str = "dev",
    force_download: bool = False
) -> str:
    """
    Downloads the CoQA dataset and returns the path to the downloaded file.
    
    Args:
        save_dir: Directory to save the dataset
        version: Dataset version ('dev' or 'train')
        force_download: If True, redownload even if file exists
        
    Returns:
        Path to the downloaded JSON file
    """
    # Create save directory if it doesn't exist
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Define URLs for dataset versions
    COQA_URLS = {
        'dev': 'https://downloads.cs.stanford.edu/nlp/data/coqa/coqa-dev-v1.0.json',
        'train': 'https://downloads.cs.stanford.edu/nlp/data/coqa/coqa-train-v1.0.json'
    }
    
    if version not in COQA_URLS:
        raise ValueError(f"Invalid version: {version}. Must be one of {list(COQA_URLS.keys())}")
    
    url = COQA_URLS[version]
    filename = url.split('/')[-1]
    save_path = save_dir / filename
    
    # Check if file already exists
    if save_path.exists() and not force_download:
        print(f"Found existing file at {save_path}")
        try:
            # Validate JSON format
            with open(save_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                print(f"Dataset contains {len(data['data'])} stories")
            return str(save_path)
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Existing file is corrupted or invalid: {e}")
            print("Will download a fresh copy...")
    
    # Download the file
    print(f"Downloading CoQA {version} dataset from {url}")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    # Get file size for progress bar
    file_size = int(response.headers.get('content-length', 0))
    
    # Download with progress bar
    with open(save_path, 'wb') as f, tqdm(
        desc=filename,
        total=file_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as pbar:
        for data in response.iter_content(chunk_size=1024):
            size = f.write(data)
            pbar.update(size)
    
    print(f"Successfully downloaded CoQA {version} dataset")
    
    # Validate JSON format
    try:
        with open(save_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            print(f"Dataset contains {len(data['data'])} stories")
    except (json.JSONDecodeError, KeyError) as e:
        raise ValueError(f"Downloaded file is not in valid CoQA format: {e}")
    
    return str(save_path)

# Download dev set
dev_path = download_coqa_dataset(version='dev')
print(f"Dataset downloaded to: {dev_path}")

# Download train set
train_path = download_coqa_dataset(version='train')
print(f"Dataset downloaded to: {train_path}")

Found existing file at /nfs/students/daro/data/CoQA/coqa-dev-v1.0.json
Dataset contains 500 stories
Dataset downloaded to: /nfs/students/daro/data/CoQA/coqa-dev-v1.0.json
Found existing file at /nfs/students/daro/data/CoQA/coqa-train-v1.0.json
Dataset contains 7199 stories
Dataset downloaded to: /nfs/students/daro/data/CoQA/coqa-train-v1.0.json


## 2.2 Load setup

In [26]:
import json
import random
from typing import List, Tuple, Dict, Optional
from pathlib import Path
# Import the typo modification function from the existing codebase
from src.reliability.apply_typos import apply_typo_modifications

def create_typo_dict(typo_type: str, intensity: int) -> Dict[str, int]:
    """
    Creates a dictionary of typo modifications with specified intensity.
    Reuses the same typo types as the original implementation.
    """
    base_dict = {
        "char_insertion": 0,
        "char_deletion": 0,
        "char_replacement": 0,
        "char_repetition": 0,
        "char_swapping": 0,
        "word_CMW": 0,
        "char_LCC": 0,
        "word_synonym": 0,
        "char_insert_noise": 0,
        "word_repeat": 0,
        "char_substitution": 0,
        "word_emoji": 0,
        "word_internet_slang": 0,
        "word_phrase_translation": 0,
        "word_context_aware_insertion": 0,
        "word_remove_punctuation": 0,
        "word_keyword_only": 0,
        "word_taxonomy_pos": 0,
        "word_taxonomy_neg": 0
    }
    
    if typo_type in base_dict:
        base_dict[typo_type] = intensity
    elif typo_type == "random":
        for _ in range(intensity):
            key = random.choice(list(base_dict.keys()))
            base_dict[key] += 1
    
    return base_dict

def load_coqa_dataset(
    file_path: str,
    max_entries: Optional[int] = None,
    typo_type: str = "none",
    typo_intensity: int = 0,
    concatenate_qa: bool = False
) -> List[Tuple[str, str, str]]:
    """
    Loads the CoQA dataset and returns a list of (story, question, answer) tuples.
    Processes the data in the same way as the semantic uncertainty script.
    
    Args:
        file_path: Path to the CoQA JSON file
        max_entries: Maximum number of QA pairs to load (None for all)
        typo_type: Type of typo to apply ("none" for no typos)
        typo_intensity: Intensity of typo modifications
        concatenate_qa: If True, concatenates previous Q&A pairs into the question string
        
    Returns:
        List of (story, question, answer) tuples
    """
    qa_triplets = []
    
    # Load the CoQA dataset
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)['data']
    
    # Create typo dictionary if needed
    typo_dict = None
    if typo_type != "none":
        typo_dict = create_typo_dict(typo_type, typo_intensity)
    
    # Process each story in the dataset
    for sample in data:
        if max_entries and len(qa_triplets) >= max_entries:
            break
            
        story = sample['story']
        questions = sample['questions']
        answers = sample['answers']
        
        if concatenate_qa:
            # Initialize the concatenated question history for this story
            qa_history = ""
            
            # Process each question for the current story
            for question_index, (question, answer) in enumerate(zip(questions, answers)):
                question_text = question['input_text']
                answer_text = answer['input_text']
                
                if typo_dict is not None:
                    question_text = apply_typo_modifications(question_text, typo_dict, [answer_text])
                
                # For the first question, just add the story and question
                if question_index == 0:
                    current_question = f"{story}. Q: {question_text}"
                else:
                    # Add the previous Q&A pair to the history and create new question
                    qa_history += f"Q: {questions[question_index-1]['input_text']}"
                    qa_history += f", A: {answers[question_index-1]['input_text']}, "
                    current_question = f"{story}. {qa_history}Q: {question_text}"
                
                qa_triplets.append((
                    story,  # Original story
                    current_question,  # Concatenated history + current question
                    answer_text  # Current answer
                ))
                
                if max_entries and len(qa_triplets) >= max_entries:
                    break
                    
        else:
            # Original behavior: process each Q&A pair separately
            current_context = story
            
            for question_index, (question, answer) in enumerate(zip(questions, answers)):
                question_text = question['input_text']
                answer_text = answer['input_text']
                
                if typo_dict is not None:
                    question_text = apply_typo_modifications(question_text, typo_dict, [answer_text])
                
                qa_triplets.append((
                    current_context,
                    question_text,
                    answer_text
                ))
                
                # Update the context with this Q&A pair for the next question
                if not current_context.endswith('.'):
                    current_context += '.'
                current_context += f" Q: {question_text} A: {answer_text}"
                
                if max_entries and len(qa_triplets) >= max_entries:
                    break
    
    return qa_triplets

def load_coqa_dataset_pairs(
    file_path: str,
    max_entries: Optional[int] = None,
    typo_type: str = "none",
    typo_intensity: int = 0
) -> List[Tuple[str, str]]:
    """
    Loads the CoQA dataset and returns a list of (question, answer) pairs,
    with previous Q&A pairs concatenated into the question string.
    
    Args:
        file_path: Path to the CoQA JSON file
        max_entries: Maximum number of QA pairs to load (None for all)
        typo_type: Type of typo to apply ("none" for no typos)
        typo_intensity: Intensity of typo modifications
        
    Returns:
        List of (question, answer) tuples with concatenated history
    """
    qa_triplets = load_coqa_dataset(
        file_path=file_path,
        max_entries=max_entries,
        typo_type=typo_type,
        typo_intensity=typo_intensity,
        concatenate_qa=True
    )
    
    return [(question, answer) for _, question, answer in qa_triplets]

## 2.3 Debugging CoQA

In [27]:
coqa_path = "/nfs/students/daro/data/CoQA/coqa-dev-v1.0.json"

# Test different loading configurations
config = {
    "max_entries": 5,
    "typo_type": "none",
    "typo_intensity": 0
}

print(f"\nTesting configuration: {config} with concatenation=False")
qa_triplets = load_coqa_dataset(
    coqa_path,
    max_entries=config['max_entries'],
    typo_type=config['typo_type'],
    typo_intensity=config['typo_intensity'],
    concatenate_qa=False
)
    
# Print the first few QA triplets
for idx, (story, question, answer) in enumerate(qa_triplets[:10]):
    print(f"\nPair {idx + 1}:")
    print(f"Story: {story[:30]}...")  # Print first 100 chars of story
    print(f"Q: {question}")
    print(f"A: {answer}")
    
print(f"\nTesting configuration: {config} with concatenation=True")
qa_pairs = load_coqa_dataset_pairs(
    coqa_path,
    max_entries=config['max_entries'],
    typo_type=config['typo_type'],
    typo_intensity=config['typo_intensity'],
)

# Print the first few QA pairs
for idx, (question, answer) in enumerate(qa_pairs[:10]):
    print(f"\nPair {idx + 1}:")
    print(f"Q: {question}")
    print(f"A: {answer}")


Testing configuration: {'max_entries': 5, 'typo_type': 'none', 'typo_intensity': 0} with concatenation=False

Pair 1:
Story: Once upon a time, in a barn ne...
Q: What color was Cotton?
A: white

Pair 2:
Story: Once upon a time, in a barn ne...
Q: Where did she live?
A: in a barn

Pair 3:
Story: Once upon a time, in a barn ne...
Q: Did she live alone?
A: no

Pair 4:
Story: Once upon a time, in a barn ne...
Q: Who did she live with?
A: with her mommy and 5 sisters

Pair 5:
Story: Once upon a time, in a barn ne...
Q: What color were her sisters?
A: orange and white

Testing configuration: {'max_entries': 5, 'typo_type': 'none', 'typo_intensity': 0} with concatenation=True

Pair 1:
Q: Once upon a time, in a barn near a farm house, there lived a little white kitten named Cotton. Cotton lived high up in a nice warm place above the barn where all of the farmer's horses slept. But Cotton wasn't alone in her little home above the barn, oh no. She shared her hay bed with her mommy and 5 other s

In [33]:
len(qa_pairs)

5

## 2.4 Loading script

In [21]:
# Run debug function
qa_triplets = debug_coqa_loading()


Testing configuration: {'max_entries': 5, 'typo_type': 'none', 'typo_intensity': 0}

Pair 1:
Story: Once upon a time, in a barn near a farm house, there lived a little white kitten named Cotton. Cotto...
Q: What color was Cotton?
A: white

Pair 2:
Story: Once upon a time, in a barn near a farm house, there lived a little white kitten named Cotton. Cotto...
Q: Where did she live?
A: in a barn

Pair 3:
Story: Once upon a time, in a barn near a farm house, there lived a little white kitten named Cotton. Cotto...
Q: Did she live alone?
A: no

Pair 4:
Story: Once upon a time, in a barn near a farm house, there lived a little white kitten named Cotton. Cotto...
Q: Who did she live with?
A: with her mommy and 5 sisters

Pair 5:
Story: Once upon a time, in a barn near a farm house, there lived a little white kitten named Cotton. Cotto...
Q: What color were her sisters?
A: orange and white

Testing configuration: {'max_entries': 3, 'typo_type': 'char_insertion', 'typo_intensity': 1}

Pair 1:
S

In [23]:
with open(coqa_path, 'r', encoding='utf-8') as f:
    data = json.load(f)['data']

# 3. Semantic Uncertainty Paper Code

In [8]:
import json

import evaluate
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

coqa_path = "/nfs/students/daro/data/CoQA/"

device_map = {
    'model.decoder.embed_tokens': 0,
    'model.decoder.embed_positions': 0,
    'model.decoder.layers.0': 0,
    'model.decoder.layers.1': 0,
    'model.decoder.layers.2': 0,
    'model.decoder.layers.3': 0,
    'model.decoder.layers.4': 0,
    'model.decoder.layers.5': 0,
    'model.decoder.layers.6': 0,
    'model.decoder.layers.7': 0,
    'model.decoder.layers.8': 0,
    'model.decoder.layers.9': 0,
    'model.decoder.layers.10': 0,
    'model.decoder.layers.11': 0,
    'model.decoder.layers.12': 0,
    'model.decoder.layers.13': 0,
    'model.decoder.layers.14': 0,
    'model.decoder.layers.15': 0,
    'model.decoder.layers.16': 0,
    'model.decoder.layers.17': 0,
    'model.decoder.layers.18': 0,
    'model.decoder.layers.19': 0,
    'model.decoder.layers.20': 0,
    'model.decoder.layers.21': 0,
    'model.decoder.layers.22': 0,
    'model.decoder.layers.23': 0,
    'model.decoder.layers.24': 0,
    'model.decoder.layers.25': 1,
    'model.decoder.layers.26': 1,
    'model.decoder.layers.27': 1,
    'model.decoder.layers.28': 1,
    'model.decoder.layers.29': 1,
    'model.decoder.layers.30': 1,
    'model.decoder.layers.31': 1,
    'model.decoder.layers.32': 1,
    'model.decoder.layers.33': 1,
    'model.decoder.layers.34': 1,
    'model.decoder.layers.35': 1,
    'model.decoder.layers.36': 1,
    'model.decoder.layers.37': 1,
    'model.decoder.layers.38': 1,
    'model.decoder.layers.39': 1,
    'model.decoder.layers.40': 1,
    'model.decoder.layers.41': 1,
    'model.decoder.layers.42': 1,
    'model.decoder.layers.43': 1,
    'model.decoder.layers.44': 1,
    'model.decoder.layers.45': 1,
    'model.decoder.layers.46': 1,
    'model.decoder.layers.47': 1,
    'model.decoder.layers.48': 1,
    'model.decoder.final_layer_norm': 1,
    'lm_head': 1
}

data_dir = ''
hf_datasets_cache = ''
output_dir = ''

with open(f'{coqa_path}/coqa-dev-v1.0.json', 'r') as infile:
    data = json.load(infile)['data']

dataset = {}

dataset['story'] = []
dataset['question'] = []
dataset['answer'] = []
dataset['additional_answers'] = []
dataset['rouge1'] = []
dataset['rouge2'] = []
dataset['rougeL'] = []
dataset['semantic_variability'] = []
dataset['id'] = []
dataset['input'] = []

for sample_id, sample in enumerate(data):
    story = sample['story']
    questions = sample['questions']
    answers = sample['answers']
    additional_answers = sample['additional_answers']
    for question_index, question in enumerate(questions):
        dataset['story'].append(story)
        dataset['question'].append(question['input_text'])
        dataset['answer'].append({
            'text': answers[question_index]['input_text'],
            'answer_start': answers[question_index]['span_start']
        })
        dataset['id'].append(sample['id'] + '_' + str(question_index))
        additional_answers_list = []

        for i in range(3):
            additional_answers_list.append(additional_answers[str(i)][question_index]['input_text'])

        dataset['additional_answers'].append(additional_answers_list)
        story = story + ' Q: ' + question['input_text'] + ' A: ' + answers[question_index]['input_text']
        if not story[-1] == '.':
            story = story + '.'
        all_answers = [answers[question_index]['input_text']] + additional_answers_list

        answer_list_1 = []
        answer_list_2 = []
        has_semantically_different_answers = False
        inputs = []

        # This computes the syntactic similarity across the reference answers
        for i, reference_answer in enumerate(all_answers):
            for j in range(4):
                if i != j:
                    answer_list_1.append(all_answers[i])
                    answer_list_2.append(all_answers[j])

                    qa_1 = question['input_text'] + ' ' + all_answers[i]
                    qa_2 = question['input_text'] + ' ' + all_answers[j]

                    input = qa_1 + ' [SEP] ' + qa_2

                    inputs.append(input)
                    dataset['input'].append(input)
                    #print(encoded_input)

        # encoded_input = tokenizer.batch_encode_plus(inputs, padding=True)

        # prediction = model(torch.tensor(encoded_input['input_ids'], device='cuda'))['logits']

        # predicted_label = torch.argmax(prediction, dim=1)
        # if 0 in predicted_label:
        #     has_semantically_different_answers = True

        # dataset['semantic_variability'].append(has_semantically_different_answers)

        # results = rouge.compute(predictions=answer_list_1, references=answer_list_2)
        # dataset['rouge1'].append(results['rouge1'].mid.fmeasure)
        # dataset['rouge2'].append(results['rouge2'].mid.fmeasure)
        # dataset['rougeL'].append(results['rougeL'].mid.fmeasure)

# dataset_df = pd.DataFrame.from_dict(dataset)

# dataset = Dataset.from_pandas(dataset_df)

In [14]:
import numpy as np
print(np.unique(dataset['input'])[:10])

['A Barcelona member launched a case against who? Rosell [SEP] A Barcelona member launched a case against who? Rosell'
 'A daughter? yes [SEP] A daughter? yes'
 'A source mentioned that Jordan will use what against the terrorists? airstrikes [SEP] A source mentioned that Jordan will use what against the terrorists? airstrikes'
 'A source mentioned that Jordan will use what against the terrorists? airstrikes [SEP] A source mentioned that Jordan will use what against the terrorists? airstrikes.'
 'A source mentioned that Jordan will use what against the terrorists? airstrikes. [SEP] A source mentioned that Jordan will use what against the terrorists? airstrikes'
 'AND THE LOCATION? in the cab [SEP] AND THE LOCATION? in the cab'
 'About what? Cooking [SEP] About what? Cooking.'
 'About what? Cooking [SEP] About what? about cooking'
 'About what? Cooking [SEP] About what? cooking'
 'About what? Cooking. [SEP] About what? Cooking']
